.pdf, .docx, .docm - DoclingDocumentParser
Реализовано: извлечение текста блоками с привязкой к странице, разбор таблиц с сохранением заголовков столбцов при каждой ячейке.
Не реализовано: постобработка текста - не убираются повторяющиеся колонтитулы/номера страниц, не склеиваются слова, разорванные переносом на конце строки, не чистятся маркеры сносок посреди предложений.

In [7]:
from pathlib import Path

SAMPLE_DIR = Path("sample_docs/synthetic_v2")
files = [p for p in SAMPLE_DIR.rglob("*") if p.is_file()]

In [8]:
from __future__ import annotations

import io
import logging
from functools import lru_cache
from pathlib import Path
from typing import Any

logger = logging.getLogger(__name__)

_SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".docm"}


@lru_cache(maxsize=1)
def _build_converter() -> Any:
    """Build one process-wide Docling converter (model load is expensive)."""
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption

    pdf_options = PdfPipelineOptions()
    pdf_options.do_ocr = False
    pdf_options.do_table_structure = True
    return DocumentConverter(
        allowed_formats=[InputFormat.PDF, InputFormat.DOCX],
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)},
    )


class DoclingDocumentParser:
    """PDF (text layer only) and DOCX parser producing provenance-carrying spans."""

    parser_profile = "docling_v1"

    def parse(self, *, content: bytes, filename: str) -> ParsedDocument:
        extension = Path(filename).suffix.lower()
        if extension not in _SUPPORTED_EXTENSIONS:
            raise ParserError(f"Unsupported parser extension: {extension}")

        from docling.datamodel.base_models import ConversionStatus, DocumentStream

        converter = _build_converter()
        stream_name = (
            f"{Path(filename).stem}.docx" if extension == ".docm" else filename
        )
        stream = DocumentStream(name=stream_name, stream=io.BytesIO(content))
        try:
            result = converter.convert(stream, raises_on_error=True)
        except Exception as error:
            raise ParserError(f"Docling failed to parse {filename}: {error}") from error
        if result.status not in (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS):
            raise ParserError(f"Docling conversion status {result.status} for {filename}")

        document = result.document
        blocks = self._collect_blocks(document)
        tables = self._collect_tables(document)
        parsed = ParsedDocument(
            blocks=blocks,
            tables=tables,
            title=self._document_title(document, filename),
            parser_profile=self.parser_profile,
        )
        if extension == ".pdf" and not parsed.has_content:
            raise NoTextLayerError(
                f"PDF {filename} has no extractable text layer (OCR is out of scope)."
            )
        if not parsed.has_content:
            raise ParserError(f"Document {filename} produced no extractable content.")
        return parsed

    def _collect_blocks(self, document: Any) -> list[ParsedBlock]:
        blocks: list[ParsedBlock] = []
        block_ordinal = 0
        for item, _level in document.iterate_items():
            text = getattr(item, "text", None)
            if not text or not str(text).strip():
                continue
            page, bbox = self._provenance(item)
            block_ordinal += 1
            locator = f"block_{block_ordinal}"
            if bbox is not None:
                locator = f"{locator}:bbox_{bbox}"
            blocks.append(ParsedBlock(text=str(text).strip(), page=page, locator=locator))
        return blocks

    def _collect_tables(self, document: Any) -> list[ParsedTable]:
        tables: list[ParsedTable] = []
        for table_index, table_item in enumerate(getattr(document, "tables", []) or [], start=1):
            page, _bbox = self._provenance(table_item)
            header, rows = self._table_rows(table_item)
            if rows:
                tables.append(
                    ParsedTable(rows=rows, table_index=table_index, page=page, header=header)
                )
        return tables

    def _table_rows(self, table_item: Any) -> tuple[list[str], list[ParsedTableRow]]:
        grid = getattr(getattr(table_item, "data", None), "grid", None)
        if not grid:
            return [], []
        header = [str(getattr(cell, "text", "") or "").strip() for cell in grid[0]] if grid else []
        rows: list[ParsedTableRow] = []
        for row_index, grid_row in enumerate(grid[1:], start=2):
            cells = [
                ParsedTableCell(
                    text=str(getattr(cell, "text", "") or "").strip(),
                    row_index=row_index,
                    col_index=col_index,
                    header=header[col_index - 1] if col_index - 1 < len(header) else "",
                )
                for col_index, cell in enumerate(grid_row, start=1)
            ]
            if any(cell.text for cell in cells):
                rows.append(ParsedTableRow(cells=cells, row_index=row_index, headers=header))
        return header, rows

    def _provenance(self, item: Any) -> tuple[int | None, str | None]:
        provenance = getattr(item, "prov", None) or []
        for entry in provenance:
            page = getattr(entry, "page_no", None)
            bbox = getattr(entry, "bbox", None)
            bbox_text: str | None = None
            if bbox is not None:
                left = getattr(bbox, "l", None)
                top = getattr(bbox, "t", None)
                if left is not None and top is not None:
                    bbox_text = f"{left:.0f}x{top:.0f}"
            if page is not None:
                return int(page), bbox_text
        return None, None

    def _document_title(self, document: Any, filename: str) -> str:
        name = getattr(document, "name", None)
        if name and str(name).strip() and str(name) != "file":
            return str(name)
        return Path(filename).stem

.doc - LegacyDocParser
Реализовано: только извлечение через antiword/catdoc, разрез на абзацы по двойному переносу строки.
Не реализовано: вообще никакой чистки текста - ни колонтитулов, ни переносов, ни сносок. Это самый сырой из всех четырёх.


In [5]:
from __future__ import annotations

import logging
import shutil
import subprocess
import tempfile
from pathlib import Path

logger = logging.getLogger(__name__)


class LegacyDocParser:
    """Legacy .doc text extraction via antiword/catdoc (no OCR, no layout)."""

    parser_profile = "legacy_doc_v1"

    def parse(self, *, content: bytes, filename: str) -> ParsedDocument:
        if Path(filename).suffix.lower() != ".doc":
            raise ParserError(f"Unsupported legacy-doc extension: {filename}")
        text = self._extract_text(content)
        if not text.strip():
            raise ParserError(
                f"Legacy .doc {filename}: no text extracted "
                "(antiword/catdoc unavailable or empty document)."
            )
        blocks = [
            ParsedBlock(text=paragraph.strip(), page=None, locator=f"block_{index}")
            for index, paragraph in enumerate(text.split("\n\n"), start=1)
            if paragraph.strip()
        ]
        return ParsedDocument(
            blocks=blocks,
            tables=[],
            title=Path(filename).stem,
            parser_profile=self.parser_profile,
        )

    def _extract_text(self, content: bytes) -> str:
        with tempfile.NamedTemporaryFile(suffix=".doc") as handle:
            handle.write(content)
            handle.flush()
            for command in (["antiword", handle.name], ["catdoc", "-w", handle.name]):
                if shutil.which(command[0]) is None:
                    continue
                result = subprocess.run(
                    command, capture_output=True, timeout=120, check=False
                )
                if result.returncode == 0 and result.stdout.strip():
                    return result.stdout.decode("utf-8", errors="ignore")
                logger.warning(
                    "%s failed on .doc: %s", command[0], result.stderr.decode()[:150]
                )
        return ""

.xlsx, .xls - SpreadsheetDocumentParser
Реализовано: разбор через pandas, привязка каждой ячейки к заголовку её столбца, лимиты на размер (число листов/строк/столбцов).
Не реализовано: проверка того что первая непустая строка листа действительно является заголовком таблицы. Сейчас код слепо берёт первую непустую строку как заголовок - если сверху листа есть строки метаданных (дата отчёта, название организации, пустые строки-разделители перед настоящей таблицей), эти строки станут ложными названиями столбцов, и все данные ниже будут подписаны неправильно.

In [6]:
from __future__ import annotations

import io
import logging
import os
from pathlib import Path
from typing import Any

logger = logging.getLogger(__name__)

_SUPPORTED_EXTENSIONS = {".xlsx", ".xls"}

_DEFAULT_MAX_SHEETS = 50
_DEFAULT_MAX_ROWS_PER_SHEET = 5000
_DEFAULT_MAX_COLUMNS = 60


def _cap(env_name: str, default: int) -> int:
    raw = os.getenv(env_name)
    if raw is None:
        return default
    try:
        value = int(raw)
    except ValueError:
        return default
    return value if value > 0 else default


class SpreadsheetDocumentParser:
    """XLSX/XLS -> table-row evidence spans with sheet/row provenance."""

    parser_profile = "spreadsheet_v1"

    def parse(self, *, content: bytes, filename: str) -> ParsedDocument:
        extension = Path(filename).suffix.lower()
        if extension not in _SUPPORTED_EXTENSIONS:
            raise ParserError(f"Unsupported spreadsheet extension: {extension}")
        import pandas as pd

        try:
            sheets: dict[str, Any] = pd.read_excel(
                io.BytesIO(content), sheet_name=None, header=None, dtype=str
            )
        except Exception as error:
            raise ParserError(f"Spreadsheet parse failed for {filename}: {error}") from error

        max_sheets = _cap("INGEST_XLSX_MAX_SHEETS", _DEFAULT_MAX_SHEETS)
        max_rows = _cap("INGEST_XLSX_MAX_ROWS", _DEFAULT_MAX_ROWS_PER_SHEET)
        max_columns = _cap("INGEST_XLSX_MAX_COLUMNS", _DEFAULT_MAX_COLUMNS)
        tables: list[ParsedTable] = []
        truncated_rows = 0
        for table_index, (sheet_name, frame) in enumerate(sheets.items(), start=1):
            if table_index > max_sheets:
                break
            frame = frame.fillna("")
            all_rows = list(frame.itertuples(index=False))
            header = [str(value).strip() for value in all_rows[0][:max_columns]] if all_rows else []
            rows: list[ParsedTableRow] = []
            for row_index, row in enumerate(all_rows[1:], start=2):
                if row_index - 1 > max_rows:
                    truncated_rows += len(all_rows) - 1 - max_rows
                    break
                cells = [
                    ParsedTableCell(
                        text=str(value).strip(),
                        row_index=row_index,
                        col_index=col_index,
                        header=header[col_index - 1] if col_index - 1 < len(header) else "",
                    )
                    for col_index, value in enumerate(row[:max_columns], start=1)
                ]
                if any(cell.text for cell in cells):
                    rows.append(ParsedTableRow(cells=cells, row_index=row_index, headers=header))
            if rows:
                tables.append(
                    ParsedTable(rows=rows, table_index=table_index, page=None, header=header)
                )
                logger.debug("Sheet %s: %d rows", sheet_name, len(rows))
        metadata: dict[str, Any] = {"sheet_count": len(sheets)}
        if truncated_rows > 0:
            metadata["truncated_rows"] = truncated_rows
        return ParsedDocument(
            blocks=[],
            tables=tables,
            title=Path(filename).stem,
            parser_profile=self.parser_profile,
            metadata=metadata,
        )

.pptx - отсутствует полностью
Нет обработчика вообще, ни один из парсеров его не берёт.